# 05 集中 GPU 验收（A → E，失败即停止）

本 Notebook 只组织阶段执行；模型命令分别由 `/content/.venv_planner/bin/python` 和 `/content/.venv/bin/python` 运行。每个阶段独立写入 ledger，前一阶段未通过时后一阶段会拒绝启动。默认不下载权重、不启动训练；D/E 的预算和命令保持未设置。

In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import Image as DisplayImage, display

PROJECT_ROOT = Path("/content/vla-3d-trajectory-guidance").resolve()
CONFIG_PATH = PROJECT_ROOT / "configs" / "gpu_acceptance_colab.json"
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = Path(os.environ.get(
    "VLA_GPU_ACCEPTANCE_RUN_DIR",
    str(PROJECT_ROOT / "outputs" / "gpu_acceptance" / timestamp),
)).resolve()
RUN_DIR.mkdir(parents=True, exist_ok=True)

def run_stage(stage):
    command = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "run_gpu_acceptance_stage.py"),
        "--stage", stage,
        "--project-root", str(PROJECT_ROOT),
        "--config", str(CONFIG_PATH),
        "--run-dir", str(RUN_DIR),
    ]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, check=True)

print(f"Run directory: {RUN_DIR}")
print("默认输出位于临时 /content；长训练开始后再把 RUN_DIR 指向已挂载的 Google Drive。")

## 阶段 A：环境、checkpoint、缓存与标定

不加载大模型。分别用两个目标解释器检查固定依赖与 CUDA，并重新读取已保存的 RGB-D、K、外参和 checkpoint manifest。

In [ ]:
run_stage("A")

In [ ]:
ledger = json.loads((RUN_DIR / "acceptance_ledger.json").read_text(encoding="utf-8"))
print(json.dumps(ledger["stages"]["A"], ensure_ascii=False, indent=2))
assert ledger["stages"]["A"]["status"] == "passed"

## 阶段 B：LIBERO 单帧真实高层规划

只在 A 通过后加载一次 3D HAMSTER。输出 UVD、camera/world XYZ、几何诊断、overlay 和统一轨迹缓存。若几何无效，阶段 B 失败并阻止 C。

In [ ]:
run_stage("B")

In [ ]:
ledger = json.loads((RUN_DIR / "acceptance_ledger.json").read_text(encoding="utf-8"))
stage_b_summary_path = Path(ledger["stages"]["B"]["artifacts"]["summary"])
stage_b_summary = json.loads(stage_b_summary_path.read_text(encoding="utf-8"))
print(json.dumps({
    "valid": stage_b_summary["valid"],
    "waypoint_count": stage_b_summary["waypoint_count"],
    "timing_seconds": stage_b_summary["timing_seconds"],
    "geometry_validation": stage_b_summary["geometry_validation"],
    "trajectory_cache_record": stage_b_summary["artifacts"]["trajectory_cache_record"],
}, ensure_ascii=False, indent=2))
overlay = Path(stage_b_summary["artifacts"]["trajectory_overlay_png"])
if overlay.is_file():
    display(DisplayImage(filename=str(overlay)))
assert stage_b_summary["valid"] is True

## 阶段 C：真实 SmolVLA E0/E1/E2 冒烟

先在本地配置 `VLA_SMOLVLA_POLICY_PATH` 和 `VLA_SMOLVLA_BATCH_ARTIFACT`。batch 必须是已预处理、已做因果检查的本地 tensor artifact，并包含 trajectory keys。本阶段只做真实 forward/backward、梯度、动作维度与显存检查，不做 optimizer step。

In [ ]:
RUN_STAGE_C = False  # 路径准备好并目视确认 B 后再改为 True
if RUN_STAGE_C:
    assert os.environ.get("VLA_SMOLVLA_POLICY_PATH"), "未设置本地 policy 路径"
    assert os.environ.get("VLA_SMOLVLA_BATCH_ARTIFACT"), "未设置 batch artifact 路径"
    run_stage("C")
else:
    print("Stage C 未启动；未加载 SmolVLA，也未下载权重。")

## 阶段 D / E：小规模验证与正式实验

D/E 默认禁用，且配置中的 command 故意为空。只有 C 的真实接口与显存检查通过后，才填写单任务 pilot；D 通过并确定正式预算后再填写 E。ledger 会再次检查先决阶段。

In [ ]:
RUN_STAGE_D = False
RUN_STAGE_E = False
if RUN_STAGE_D:
    run_stage("D")
if RUN_STAGE_E:
    run_stage("E")
if not RUN_STAGE_D and not RUN_STAGE_E:
    print("D/E 未启动；没有训练或闭环 rollout。")

In [ ]:
ledger = json.loads((RUN_DIR / "acceptance_ledger.json").read_text(encoding="utf-8"))
print(json.dumps({stage: record["status"] for stage, record in ledger["stages"].items()}, ensure_ascii=False, indent=2))